In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
from scipy.signal import butter, filtfilt
from scipy.ndimage import uniform_filter1d

In [2]:
HMD_dataframe, truth_dataframe = pd.read_csv("HMD/Messung1_160-170W_58-64rpm.csv"), pd.read_csv("HMD/activity1_19603463475.csv")
#HMD_dataframe, truth_dataframe = pd.read_csv("HMD/Messung2_180-20-2300W_58-80rpm.csv"), pd.read_csv("HMD/activity2_19603533361.csv")
#HMD_dataframe, truth_dataframe = pd.read_csv("HMD/Messung3_5x30sec160+30secPEAK.csv"), pd.read_csv("HMD/activity3_19603611215.csv")

In [3]:
truth_dataframe['Time'] = pd.to_datetime(truth_dataframe['Time'])
truth_dataframe['Time'] = (truth_dataframe['Time'] - truth_dataframe['Time'].iloc[0]).dt.total_seconds()
truth_dataframe['Watts'] = truth_dataframe['Watts'].astype(int)
truth_dataframe['Cadence'] = truth_dataframe['Cadence'].astype(int)

print(truth_dataframe.head(5))
average_watts = truth_dataframe['Watts'].mean()
print(f"Average Watts value: {average_watts:.2f}")


   Time  Watts  Cadence
0   0.0      5        0
1   1.0     29        0
2   2.0     64       22
3   3.0    179       28
4   4.0    158       35
Average Watts value: 154.18


In [4]:
def plot_sensor_data(adc_df, mpu_df, truth_dataframe=None, plot_filtered_adc=False):
    """
    Plot sensor data. If plot_filtered_adc is True, plot adc0_filtered and adc1_filtered below ADC0 and ADC1.
    """

    # Determine number of rows
    n_rows = 6 + (1 if truth_dataframe is not None else 0)
    if plot_filtered_adc and 'adc0_filtered' in adc_df:
        n_rows += 1
    if plot_filtered_adc and 'adc1_filtered' in adc_df:
        n_rows += 1

    # Build subplot titles
    subplot_titles = ["Piezo_1"]
    if plot_filtered_adc and 'adc0_filtered' in adc_df:
        subplot_titles.append("Piezo_1 Filtered")
    subplot_titles.append("Piezo_2")
    if plot_filtered_adc and 'adc1_filtered' in adc_df:
        subplot_titles.append("Piezo_2 Filtered")
    subplot_titles += ["ADC2", "ADC3", "Accelerometer", "Gyroscope"]
    if truth_dataframe is not None:
        subplot_titles.append("Ground Truth: Watts vs Time")

    fig = sp.make_subplots(
        rows=n_rows, cols=1, shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=tuple(subplot_titles)
    )

    row = 1
    # ADC0
    t0_col = "t0"
    v0_col = "adc0"
    valid0 = adc_df[[t0_col, v0_col]].dropna()
    times0 = (valid0[t0_col].values - min(adc_df[[f"t{j}" for j in range(4)]].min().min(), mpu_df["t4"].min())) / 1_000_000.0
    fig.add_trace(go.Scatter(x=times0, y=valid0[v0_col].values, mode="lines", name="Piezo_1"), row=row, col=1)
    row += 1

    # ADC0 Filtered
    if plot_filtered_adc and 'adc0_filtered' in adc_df:
        fig.add_trace(go.Scatter(x=times0, y=adc_df.loc[valid0.index, 'adc0_filtered'], mode="lines", name="Piezo_1 Filtered"), row=row, col=1)
        row += 1

    # ADC1
    t1_col = "t1"
    v1_col = "adc1"
    valid1 = adc_df[[t1_col, v1_col]].dropna()
    times1 = (valid1[t1_col].values - min(adc_df[[f"t{j}" for j in range(4)]].min().min(), mpu_df["t4"].min())) / 1_000_000.0
    fig.add_trace(go.Scatter(x=times1, y=valid1[v1_col].values, mode="lines", name="Piezo_2"), row=row, col=1)
    row += 1

    # ADC1 Filtered
    if plot_filtered_adc and 'adc1_filtered' in adc_df:
        fig.add_trace(go.Scatter(x=times1, y=adc_df.loc[valid1.index, 'adc1_filtered'], mode="lines", name="Piezo_2 Filtered"), row=row, col=1)
        row += 1

    # ADC2
    t2_col = "t2"
    v2_col = "adc2"
    valid2 = adc_df[[t2_col, v2_col]].dropna()
    times2 = (valid2[t2_col].values - min(adc_df[[f"t{j}" for j in range(4)]].min().min(), mpu_df["t4"].min())) / 1_000_000.0
    fig.add_trace(go.Scatter(x=times2, y=valid2[v2_col].values, mode="lines", name="ADC2"), row=row, col=1)
    row += 1

    # ADC3
    t3_col = "t3"
    v3_col = "adc3"
    valid3 = adc_df[[t3_col, v3_col]].dropna()
    times3 = (valid3[t3_col].values - min(adc_df[[f"t{j}" for j in range(4)]].min().min(), mpu_df["t4"].min())) / 1_000_000.0
    fig.add_trace(go.Scatter(x=times3, y=valid3[v3_col].values, mode="lines", name="ADC3"), row=row, col=1)
    row += 1

    # Prepare MPU data
    mpu_times = mpu_df["t4"].dropna().values
    mpu_times = (mpu_times - min(adc_df[[f"t{j}" for j in range(4)]].min().min(), mpu_df["t4"].min())) / 1_000_000.0

    acc_x = mpu_df["acc_x"].values
    acc_y = mpu_df["acc_y"].values
    acc_z = mpu_df["acc_z"].values

    gyr_x = mpu_df["gyro_x"].values
    gyr_y = mpu_df["gyro_y"].values
    gyr_z = mpu_df["gyro_z"].values

    # Accelerometer
    fig.add_trace(go.Scatter(x=mpu_times, y=acc_x, mode="lines", name="Accel X"), row=row, col=1)
    fig.add_trace(go.Scatter(x=mpu_times, y=acc_y, mode="lines", name="Accel Y"), row=row, col=1)
    fig.add_trace(go.Scatter(x=mpu_times, y=acc_z, mode="lines", name="Accel Z"), row=row, col=1)
    row += 1

    # Gyroscope
    fig.add_trace(go.Scatter(x=mpu_times, y=gyr_x, mode="lines", name="Gyro X"), row=row, col=1)
    fig.add_trace(go.Scatter(x=mpu_times, y=gyr_y, mode="lines", name="Gyro Y"), row=row, col=1)
    fig.add_trace(go.Scatter(x=mpu_times, y=gyr_z, mode="lines", name="Gyro Z"), row=row, col=1)
    row += 1

    # Add truth dataframe as last subplot if provided
    if truth_dataframe is not None:
        fig.add_trace(go.Scatter(
            x=truth_dataframe["Time"],
            y=truth_dataframe["Watts"],
            mode="lines",
            name="Truth Watts"
        ), row=row, col=1)
        fig.update_yaxes(title_text="Watts", row=row, col=1)

    fig.update_layout(
        height=300 * n_rows,
        width=900,
        title_text="Sensor Data Visualization",
        showlegend=True
    )
    fig.update_xaxes(title_text="Time (s)", row=row, col=1)
    fig.show()




In [5]:
def trim_slumps_to_truth_duration(adc_df, mpu_df, truth_dataframe, threshold=200, min_active_duration=5):
    """
    Trim to the high-activity region whose duration is closest to the truth duration.
    Allows small slumps in between.
    """
    import numpy as np
    from scipy.ndimage import label

    truth_duration = truth_dataframe['Time'].iloc[-1]

    gyro_mag = np.sqrt(
        mpu_df["gyro_x"]**2 + mpu_df["gyro_y"]**2 + mpu_df["gyro_z"]**2
    ).values

    low_mask = gyro_mag <= threshold

    # Label contiguous low-activity regions
    labeled, num_features = label(low_mask)
    if num_features < 2:
        return adc_df, mpu_df

    # Get all (start, end) indices for each slump
    slumps = []
    for i in range(1, num_features + 1):
        idx = np.where(labeled == i)[0]
        if len(idx) >= min_active_duration:
            slumps.append((idx[0], idx[-1]))

    if len(slumps) < 2:
        return adc_df, mpu_df

    # Find the pair of slumps with a gap closest to the truth duration (allowing slumps in between)
    best_pair = None
    min_diff = float('inf')
    for i in range(len(slumps)):
        for j in range(i+1, len(slumps)):
            start_idx = slumps[i][1] + 1
            end_idx = slumps[j][0] - 1
            if start_idx >= end_idx:
                continue
            start_time = mpu_df["t4"].iloc[start_idx]
            end_time = mpu_df["t4"].iloc[end_idx]
            duration = (end_time - start_time) / 1_000_000.0  # microseconds to seconds
            diff = abs(duration - truth_duration)
            if diff < min_diff:
                min_diff = diff
                best_pair = (start_idx, end_idx)

    if best_pair is None:
        return adc_df, mpu_df

    # Subtract 1 second (1_000_000 microseconds) from start_time, but not below the minimum t4
    start_time = mpu_df["t4"].iloc[best_pair[0]]  # 5 seconds before the start of the slump
    min_time = mpu_df["t4"].min()
    if start_time < min_time:
        start_time = min_time

    end_time = mpu_df["t4"].iloc[best_pair[1]] - 1_000_000

    # Trim ADC and MPU data
    for i in range(4):
        t_col = f"t{i}"
        adc_df = adc_df[adc_df[t_col].isna() | ((adc_df[t_col] >= start_time) & (adc_df[t_col] <= end_time))]
    mpu_df = mpu_df[(mpu_df["t4"] >= start_time) & (mpu_df["t4"] <= end_time)]

    return adc_df, mpu_df


In [6]:
# --- Separate ADC and MPU rows ---
adc_df = HMD_dataframe[HMD_dataframe["type"] == "adc"].copy()
mpu_df = HMD_dataframe[HMD_dataframe["type"] == "mpu"].copy()

#plot_sensor_data(adc_df, mpu_df)
#plot_sensor_data(adc_df, mpu_df, truth_dataframe=truth_dataframe, plot_filtered_adc=False)


# 1. Trim start/end slumps to match truth duration
adc_df_trimmed, mpu_df_trimmed = trim_slumps_to_truth_duration(adc_df, mpu_df, truth_dataframe, threshold=200)

# 2. Plot the result
#plot_sensor_data(adc_df_trimmed, mpu_df_trimmed, truth_dataframe=truth_dataframe)

t_sec = mpu_df['t4'].dropna().values / 1_000_000  # convert to seconds
time_span = t_sec[-1] - t_sec[0]
fs = len(t_sec) / time_span
print(f"Sampling rate of mpu_df: {fs:.2f} Hz")
print(mpu_df.head(10))


Sampling rate of mpu_df: 21.06 Hz
  type  t0  adc0  t1  adc1  t2  adc2  t3  adc3           t4   acc_x    acc_y  \
0  mpu NaN   NaN NaN   NaN NaN   NaN NaN   NaN  468593775.0  2888.0  16140.0   
1  mpu NaN   NaN NaN   NaN NaN   NaN NaN   NaN  468596202.0  2888.0  16140.0   
2  mpu NaN   NaN NaN   NaN NaN   NaN NaN   NaN  468598530.0  2888.0  16140.0   
3  mpu NaN   NaN NaN   NaN NaN   NaN NaN   NaN  468600849.0  2888.0  16140.0   
4  mpu NaN   NaN NaN   NaN NaN   NaN NaN   NaN  468603163.0  2888.0  16140.0   
5  mpu NaN   NaN NaN   NaN NaN   NaN NaN   NaN  468605456.0  2888.0  16140.0   
6  mpu NaN   NaN NaN   NaN NaN   NaN NaN   NaN  468607763.0  2888.0  16140.0   
7  mpu NaN   NaN NaN   NaN NaN   NaN NaN   NaN  468610059.0  2888.0  16140.0   
8  mpu NaN   NaN NaN   NaN NaN   NaN NaN   NaN  468612359.0  2888.0  16140.0   
9  mpu NaN   NaN NaN   NaN NaN   NaN NaN   NaN  468614648.0  2888.0  16140.0   

     acc_z           t5  gyro_x  gyro_y  gyro_z  
0 -32210.0  468593775.0    -6.0   -

In [7]:
def calculate_sample_rate(adc_df):
    t0_sec = adc_df['t0'].dropna().values / 1_000_000
    time_span = t0_sec[-1] - t0_sec[0]
    fs = len(t0_sec) / time_span
    #print(f"Sample rate for ADC0: {fs:.2f} Hz")
    return fs


In [8]:


def bandpass_filter(signal, lowcut_hz, highcut_hz, fs_hz, order=4):
    """
    Apply a Butterworth bandpass filter.
    signal: numpy array or pandas Series
    lowcut_hz: lower cutoff frequency in Hz
    highcut_hz: upper cutoff frequency in Hz
    fs_hz: sampling frequency in Hz
    order: filter order
    """
    nyquist = 0.5 * fs_hz
    low = lowcut_hz / nyquist
    high = highcut_hz / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

fs = calculate_sample_rate(adc_df_trimmed)
print(f"Sample rate for ADC0: {fs:.4f} Hz")
lowcut = 0.5
highcut = 5.0

adc_df_trimmed['adc0_filtered'] = bandpass_filter(adc_df_trimmed['adc0'].dropna(), lowcut, highcut, fs)
adc_df_trimmed['adc1_filtered'] = bandpass_filter(adc_df_trimmed['adc1'].dropna(), lowcut, highcut, fs)

plot_sensor_data(adc_df_trimmed, mpu_df_trimmed, truth_dataframe=truth_dataframe, plot_filtered_adc=True)


Sample rate for ADC0: 22.3198 Hz


In [ ]:
mpu_df_trimmed['t4'] = mpu_df_trimmed['t4'] - mpu_df_trimmed['t4'].iloc[0]  # Normalize time to start at 0

mpu_time = mpu_df_trimmed['t4'] / 1_000_000  # convert to seconds

fig = sp.make_subplots(rows=1, cols=1, shared_xaxes=True, subplot_titles=[
 "Gyroscope (X, Y, Z)"
])

# Accelerometer subplot
#fig.add_trace(go.Scatter(x=mpu_time, y=mpu_df_trimmed['acc_x'], mode='lines', name='Accel X'), row=1, col=1)
#fig.add_trace(go.Scatter(x=mpu_time, y=mpu_df_trimmed['acc_y'], mode='lines', name='Accel Y'), row=1, col=1)
#fig.add_trace(go.Scatter(x=mpu_time, y=mpu_df_trimmed['acc_z'], mode='lines', name='Accel Z'), row=1, col=1)

# Gyroscope subplot
fig.add_trace(go.Scatter(x=mpu_time, y=mpu_df_trimmed['gyro_x'], mode='lines', name='Gyro X'), row=1, col=1)
fig.add_trace(go.Scatter(x=mpu_time, y=mpu_df_trimmed['gyro_y'], mode='lines', name='Gyro Y'), row=1, col=1)
fig.add_trace(go.Scatter(x=mpu_time, y=mpu_df_trimmed['gyro_z'], mode='lines', name='Gyro Z'), row=1, col=1)

fig.update_layout(
    height=700,
    xaxis_title="Time (s)",
    yaxis_title="Sensor Value",
    showlegend=True
)
fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.show()

C:\Users\Jeremias\AppData\Local\Temp\ipykernel_11308\1690436833.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Exception: The (row, col) pair sent is out of range. Use Figure.print_grid to view the subplot grid. 

In [10]:


gyro_x = uniform_filter1d(mpu_df_trimmed['gyro_x'].values, size=5)
#gyro_x = mpu_df_trimmed['gyro_x'].values
timestamps = mpu_df_trimmed['t4'].values / 1_000_000  # convert to seconds

threshold = 3  # Minimum difference to count as a maxima
maxima_indices = []
for i in range(1, len(gyro_x) - 1):
    if (gyro_x[i] > gyro_x[i - 1] + threshold) and (gyro_x[i] > gyro_x[i + 1] + threshold):
        maxima_indices.append(i)

min_peak_interval = 0.2  # seconds between valid peaks
filtered_maxima_indices = []

for idx in maxima_indices:
    if not filtered_maxima_indices:
        filtered_maxima_indices.append(idx)
    else:
        last_t = timestamps[filtered_maxima_indices[-1]]
        if timestamps[idx] - last_t >= min_peak_interval:
            filtered_maxima_indices.append(idx)

maxima_indices = filtered_maxima_indices

# Calculate time to next maxima and rpm
time_to_next_maxima = []
for i in range(len(maxima_indices) - 1):
    t_start = timestamps[maxima_indices[i]]
    t_next = timestamps[maxima_indices[i + 1]]
    seconds = t_next - t_start
    rpm = int(np.round(60 / seconds if seconds != 0 else 0))
    time_to_next_maxima.append({
        'timestamp': t_start,
        'time_to_next_maxima': seconds,
        'rpm': rpm
    })

maxima_df = pd.DataFrame(time_to_next_maxima)
maxima_df["rpm"] = uniform_filter1d(maxima_df['rpm'].values, size=5)
#print(maxima_df.head(15))

In [11]:
# Ensure timestamps are floats in seconds
maxima_df['timestamp'] = maxima_df['timestamp'].astype(float)

start_time = int(np.floor(maxima_df['timestamp'].min()))
end_time = int(np.ceil(maxima_df['timestamp'].max()))

resampled = []
for t in range(start_time, end_time + 1):
    before = maxima_df[maxima_df['timestamp'] <= t].tail(1)
    after = maxima_df[maxima_df['timestamp'] >= t].head(1)
    
    if before.empty or after.empty:
        continue
    
    # Average the RPMs of before and after
    avg_rpm = (before['rpm'].values[0] + after['rpm'].values[0]) / 2
    resampled.append({'second': t, 'avg_rpm': avg_rpm})

rpm_per_second_df = pd.DataFrame(resampled)

first_second = rpm_per_second_df['second'].min() - 2  # two seconds before the first one
extra_rows = pd.DataFrame({
    'second': [first_second, first_second + 1],
    'avg_rpm': [0, 0]
})

# Prepend and reindex
rpm_per_second_df = pd.concat([extra_rows, rpm_per_second_df], ignore_index=True)
rpm_per_second_df = rpm_per_second_df.sort_values('second').reset_index(drop=True)

# Now set last 10 entries to zero
if len(rpm_per_second_df) >= 10:
    rpm_per_second_df.loc[rpm_per_second_df.tail(10).index, 'avg_rpm'] = 0

rpm_per_second_df['second'] = rpm_per_second_df['second'] - rpm_per_second_df['second'].iloc[0]


#print(rpm_per_second_df.head())

In [12]:
# Ensure both time axes start at 0 and are in seconds
truth_time = truth_dataframe['Time'] - truth_dataframe['Time'].iloc[0]
maxima_time = rpm_per_second_df['second'] 

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=maxima_time, y=rpm_per_second_df['avg_rpm'],
    mode='lines', name='Estimated RPM (gyro maxima)'
))
fig.add_trace(go.Scatter(
    x=truth_time, y=truth_dataframe['Cadence'],
    mode='lines', name='Ground Truth Cadence'
))
fig.update_layout(
    title="Estimated RPM vs Ground Truth Cadence",
    xaxis_title="Time (s)",
    yaxis_title="RPM / Cadence"
)
fig.show()

In [13]:
# Find intervals where gyro_x is above zero and the crossing times
gyro_x = mpu_df_trimmed['gyro_x'].values
timestamps = mpu_df_trimmed['t4'].values / 1_000_000  # seconds
timestamps = timestamps - timestamps[0]  # Normalize to start at 0

crossings = []
intervals = []

above = gyro_x > 0
start_idx = None

for i in range(1, len(gyro_x)):
    if not above[i-1] and above[i]:
        # Crossing from below to above zero
        t_cross = np.interp(0, [gyro_x[i-1], gyro_x[i]], [timestamps[i-1], timestamps[i]])
        crossings.append({'type': 'up', 'time': t_cross})
        start_idx = i
    elif above[i-1] and not above[i]:
        # Crossing from above to below zero
        t_cross = np.interp(0, [gyro_x[i-1], gyro_x[i]], [timestamps[i-1], timestamps[i]])
        crossings.append({'type': 'down', 'time': t_cross})
        if start_idx is not None:
            intervals.append({'start': timestamps[start_idx], 'end': t_cross})
            start_idx = None

# If the last interval is still open and ends above zero
if above[-1] and start_idx is not None:
    intervals.append({'start': timestamps[start_idx], 'end': timestamps[-1]})

crossings_df = pd.DataFrame(crossings)
intervals_df = pd.DataFrame(intervals)

#print(crossings_df.head())
#print(intervals_df.head())

In [14]:
# Get intervals where gyro_x > 0 (from previous code)
# intervals_df contains 'start' and 'end' in seconds

adc_times = adc_df_trimmed['t0'].dropna().values / 1_000_000  # seconds
adc_times = adc_times - adc_times[0]  # Normalize to start at 0
adc_times1 = adc_df_trimmed['t1'].dropna().values / 1_000_000  # seconds
adc_times1 = adc_times1 - adc_times1[0]  # Normalize to start at 0
adc_values = adc_df_trimmed['adc0_filtered'].dropna().values
adc_values_1 = adc_df_trimmed['adc1_filtered'].dropna().values

selected_adc_values = []
selected_adc_times = []

for _, interval in intervals_df.iterrows():
    interval_start = interval['start']
    interval_end = interval['end']
    mask = (adc_times >= interval_start) & (adc_times <= interval_end)
    times_in_interval = adc_times[mask]
    times_in_interval1 = adc_times1[mask]
    times_in_interval = np.append(times_in_interval, times_in_interval1)
    values_in_interval = adc_values[mask]
    values_in_interval_1 = adc_values_1[mask]
    values_in_interval = np.append(values_in_interval,values_in_interval_1)
    selected_adc_times.append(times_in_interval)
    selected_adc_values.append(values_in_interval)


# selected_adc_times and selected_adc_values are lists of arrays, one per interval
#print(selected_adc_times[4], selected_adc_values[4])

In [15]:
# --- Stroke-based integration ---
stroke_powers = []
stroke_times = []

for i, (times, values) in enumerate(zip(selected_adc_times, selected_adc_values)):
    if len(times) < 2:
        continue  # skip if not enough samples
    if np.all(times == times[0]):
        continue  # All timestamps identical → skip

    sort_idx = np.argsort(times)
    times = times[sort_idx]
    values = values[sort_idx]


    # Integrate piezo signal over time -> force proxy
    force_proxy = (np.abs(np.trapz(values, times))*25) + 145

    # Get cadence for this stroke
    if "cadence_rpm" in intervals_df.columns:
        rpm = intervals_df.iloc[i]["cadence_rpm"]
    else:
        rpm = np.interp(
        times.mean(),
        rpm_per_second_df['second'].to_numpy(),
        rpm_per_second_df['avg_rpm'].to_numpy()
        )

    # Convert rpm to angular velocity (rad/s)
    omega = rpm * 2 * np.pi / 60

    # Torque proxy = force proxy × crank length
    crank_length = 0.18  # meters, adjust if you know the actual value
    torque_proxy = force_proxy * crank_length

    # Power = torque × angular velocity
    power_proxy = torque_proxy * omega 

    stroke_powers.append(power_proxy)
    stroke_times.append(times.mean())  # midpoint time of the stroke

# Create dataframe for plotting
stroke_power_df = pd.DataFrame({
    "time": stroke_times,
    "watts_est": stroke_powers
})

# Resample to 1-second intervals for easier comparison
stroke_power_per_sec = stroke_power_df.groupby(stroke_power_df['time'].astype(int), as_index=False)['watts_est'].mean()

sec_axis = np.arange(int(min(stroke_times)), int(max(stroke_times)) + 1)

# Interpolate watts_est to these full-second timestamps
watts_est_interp = np.interp(sec_axis, stroke_power_df['time'], stroke_power_df['watts_est'])

# Build final dataframe
stroke_power_per_sec = pd.DataFrame({
    'second': sec_axis,
    'watts_est': watts_est_interp
})

In [16]:
fig = go.Figure()

# Estimated power
fig.add_trace(go.Scatter(
    x=stroke_power_per_sec["second"],
    y=stroke_power_per_sec["watts_est"],
    mode='lines',
    name='Watts estimated'
))

# Truth data (if time aligned)

fig.add_trace(go.Scatter(
    x=truth_dataframe["Time"],
    y=truth_dataframe["Watts"],
    mode='lines',
    name='Watts Truth'
))


fig.update_layout(
    title="Power Comparison",
    xaxis_title="Time (s)",
    yaxis_title="Power (W)",
)

fig.show()


In [17]:
n = len(truth_dataframe) - len(stroke_power_per_sec)
truth_dataframe.drop(truth_dataframe.tail(n).index,inplace = True)

print("HMD Experiment 1, Piezo Sensors")

from scipy.stats import pearsonr
corr, pval = pearsonr(stroke_power_per_sec["watts_est"], truth_dataframe["Watts"])
print(f"Pearson correlation: {corr:.4f}, p-value: {pval:.9f}")

from scipy.stats import spearmanr
corr, pval = spearmanr(stroke_power_per_sec["watts_est"], truth_dataframe["Watts"])

print(f"Spearman correlation: {corr:.4f}, p-value: {pval:.9f}")

mae = np.mean(np.abs(stroke_power_per_sec["watts_est"] - truth_dataframe["Watts"]))
print(f"Mean Absolute Error: {mae:.4f} W")

rmse = np.sqrt(np.mean((stroke_power_per_sec["watts_est"] - truth_dataframe["Watts"])**2))
print(f"Root Mean Square Error: {rmse:.4f} W")

from dtaidistance import dtw
x = stroke_power_per_sec["watts_est"].interpolate().to_numpy().reshape(-1, 1)
y = truth_dataframe["Watts"].interpolate().to_numpy().reshape(-1, 1)

distance, paths = dtw.warping_paths(
    x.flatten(), 
    y.flatten(), 
    window=None, 
    psi=0
)

print("DTW distance:", distance)

HMD Experiment 1, Piezo Sensors
Pearson correlation: 0.9235, p-value: 0.000000000
Spearman correlation: 0.5057, p-value: 0.000000000
Mean Absolute Error: 8.4492 W
Root Mean Square Error: 14.5571 W
DTW distance: 175.52406136537763


In [18]:


def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

fs = calculate_sample_rate(adc_df_trimmed)
cutoff = 5.0  # Hz, adjust as needed

adc_df_trimmed['adc2'] = butter_lowpass_filter(adc_df_trimmed['adc2'].values, cutoff, fs)
adc_df_trimmed['adc3'] = butter_lowpass_filter(adc_df_trimmed['adc3'].values, cutoff, fs)

force_proxy_md = (adc_df_trimmed['adc2'] + adc_df_trimmed['adc3']) 

adc_df_trimmed['second'] = (adc_df_trimmed['t0'].values / 1_000_000 - adc_df_trimmed['t0'].values[0] / 1_000_000).astype(int)
force_per_sec = adc_df_trimmed.groupby('second', as_index=False).agg({
    'adc2': 'mean',
    'adc3': 'mean'
})

# Average the two pressure sensors
force_per_sec['force_proxy_md'] = (force_per_sec['adc2'] + force_per_sec['adc3']) / 2

# Merge with rpm data
merged = pd.merge(force_per_sec, rpm_per_second_df, on='second', how='inner')

# Calculate omega (rad/s)
merged['omega'] = merged['avg_rpm'] * 2 * np.pi / 60
crank_length = 0.18  # meters


scale_factor = 185.0  
merged['power_proxy_md'] = ((merged['force_proxy_md'] * 15 + 135) * crank_length * merged['omega']) 


In [19]:

fig = go.Figure()

# Estimated power

fig.add_trace(go.Scatter(
    x=stroke_power_per_sec["second"],
    y=stroke_power_per_sec["watts_est"],
    mode='lines',
    name='Watts estimated Piezo Sensors'
))

# Truth data (if time aligned)

fig.add_trace(go.Scatter(
    x=truth_dataframe["Time"],
    y=truth_dataframe["Watts"],
    mode='lines',
    name='Watts Truth'
))

fig.add_trace(go.Scatter(
    x=merged["second"],
    y=merged["power_proxy_md"],
    mode='lines',
    name='Watts estimated Digital Force sensors'
))


fig.update_layout(
    xaxis_title="Time (s)",
    yaxis_title="Power (W)",
)

fig.update_layout(legend=dict(
    yanchor="top",
    y=1.30,
    xanchor="left",
    x=0.01
))

fig.show()

In [20]:
n = len(truth_dataframe) - len(merged)
truth_dataframe.drop(truth_dataframe.tail(n).index,inplace = True)

print("HMD Experiment 1, Digital Force Sensors")

from scipy.stats import pearsonr
corr, pval = pearsonr(merged["power_proxy_md"], truth_dataframe["Watts"])
print(f"Pearson correlation: {corr:.4f}, p-value: {pval:.9f}")

from scipy.stats import spearmanr
corr, pval = spearmanr(merged["power_proxy_md"], truth_dataframe["Watts"])

print(f"Spearman correlation: {corr:.4f}, p-value: {pval:.9f}")

mae = np.mean(np.abs(merged["power_proxy_md"] - truth_dataframe["Watts"]))
print(f"Mean Absolute Error: {mae:.4f} W")

rmse = np.sqrt(np.mean((merged["power_proxy_md"] - truth_dataframe["Watts"])**2))
print(f"Root Mean Square Error: {rmse:.4f} W")

from dtaidistance import dtw
x = merged["power_proxy_md"].interpolate().to_numpy().reshape(-1, 1)
y = truth_dataframe["Watts"].interpolate().to_numpy().reshape(-1, 1)

distance, paths = dtw.warping_paths(
    x.flatten(), 
    y.flatten(), 
    window=None, 
    psi=0
)

print("DTW distance:", distance)

HMD Experiment 1, Digital Force Sensors
Pearson correlation: 0.8136, p-value: 0.000000000
Spearman correlation: 0.3193, p-value: 0.000000006
Mean Absolute Error: 8.7727 W
Root Mean Square Error: 21.0401 W
DTW distance: 143.25088762267666


In [21]:
"""
HMD Experiment 1, Piezo Sensors
Pearson correlation: 0.9234, p-value: 0.000000000
Spearman correlation: 0.5029, p-value: 0.000000000
Mean Absolute Error: 8.4568 W
Root Mean Square Error: 14.5620 W
DTW distance: 175.48869739514444

HMD Experiment 1, Digital Force Sensors
Pearson correlation: 0.8136, p-value: 0.000000000
Spearman correlation: 0.3209, p-value: 0.000000005
Mean Absolute Error: 8.7812 W
Root Mean Square Error: 21.0417 W
DTW distance: 143.2176699765313


HMD Experiment 2, Piezo Sensors
Pearson correlation: 0.9449, p-value: 0.000000000
Spearman correlation: 0.7698, p-value: 0.000000000
Mean Absolute Error: 11.8494 W
Root Mean Square Error: 18.7257 W
DTW distance: 193.72713576957912

HMD Experiment 2, Digital Force Sensors
Pearson correlation: 0.8937, p-value: 0.000000000
Spearman correlation: 0.7727, p-value: 0.000000000
Mean Absolute Error: 13.7091 W
Root Mean Square Error: 23.9162 W
DTW distance: 174.50170702257248



HMD Experiment 3, Piezo Sensors
Pearson correlation: 0.9287, p-value: 0.000000000
Spearman correlation: 0.8985, p-value: 0.000000000
Mean Absolute Error: 21.7066 W
Root Mean Square Error: 35.0763 W
DTW distance: 533.3955505818802


HMD Experiment 3, Digital Force Sensors
Pearson correlation: 0.8828, p-value: 0.000000000
Spearman correlation: 0.8567, p-value: 0.000000000
Mean Absolute Error: 24.2433 W
Root Mean Square Error: 41.9780 W
DTW distance: 560.6772012962523

"""

print("All HMD experiments mean, Piezo Sensors")

print(f"Pearson correlation: {np.mean([0.9234,0.9449,0.9287]):.4f}, p-value: {np.mean([0.000000000, 0.000000000, 0.000000000]):.4f}")

print(f"Spearman correlation: {np.mean([0.5029,0.7698,0.8985]):.4f}, p-value: {np.mean([0.000000000, 0.000000000, 0.000000000]):.4f}")

print(f"Mean Absolute Error: {np.mean([8.4568, 11.8494, 21.7066]):.4f} W")

print(f"Root Mean Square Error: {np.mean([14.5620, 18.7257, 35.0763]):.4f} W")

print("DTW distance:", np.mean([175.48869739514444, 193.72713576957912, 533.3955505818802]))

print()
print()

print("All HMD experiments mean, Digital Force Sensors")

print(f"Pearson correlation: {np.mean([0.8136,0.8937,0.8828]):.4f}, p-value: {np.mean([0.000000000, 0.000000000, 0.000000000]):.4f}")

print(f"Spearman correlation: {np.mean([0.3209,0.7727, 0.8567]):.4f}, p-value: {np.mean([0.000000000, 0.000000000, 0.000000000]):.4f}")

print(f"Mean Absolute Error: {np.mean( [8.7812, 13.7091, 24.2433]):.4f} W")

print(f"Root Mean Square Error: {np.mean([21.0417, 23.9162, 41.9780]):.4f} W")

print("DTW distance:", np.mean([143.2176699765313, 174.50170702257248, 560.6772012962523]))

All HMD experiments mean, Piezo Sensors
Pearson correlation: 0.9323, p-value: 0.0000
Spearman correlation: 0.7237, p-value: 0.0000
Mean Absolute Error: 14.0043 W
Root Mean Square Error: 22.7880 W
DTW distance: 300.8704612488679


All HMD experiments mean, Digital Force Sensors
Pearson correlation: 0.8634, p-value: 0.0000
Spearman correlation: 0.6501, p-value: 0.0000
Mean Absolute Error: 15.5779 W
Root Mean Square Error: 28.9786 W
DTW distance: 292.7988594317854


In [22]:
fig = go.Figure()

# Estimated power

fig.add_trace(go.Scatter(
    x=stroke_power_per_sec["second"],
    y=stroke_power_per_sec["watts_est"],
    mode='lines',
    name='Watts estimated Piezo Sensors'
))

# Truth data (if time aligned)

fig.add_trace(go.Scatter(
    x=maxima_time, y=rpm_per_second_df['avg_rpm'],
    mode='lines', name='Estimated RPM (gyro maxima)'
))

fig.add_trace(go.Scatter(
    x=merged["second"],
    y=merged["power_proxy_md"],
    mode='lines',
    name='Watts estimated Digital Force sensors'
))


fig.update_layout(
    xaxis_title="Time (s)",
    yaxis_title="Power (W)",
)

fig.update_layout(legend=dict(
    yanchor="top",
    y=1.30,
    xanchor="left",
    x=0.01
))

fig.show()

In [23]:

# Convert time to seconds
adc_df_trimmed['t0'] = adc_df_trimmed['t0'] / 1_000_000
adc_df['t0'] = adc_df['t0'] / 1_000_000
adc_df_trimmed['t0'] = adc_df_trimmed['t0'] - adc_df_trimmed['t0'].iloc[0] + 85
adc_df['t0'] = adc_df['t0'] - adc_df['t0'].iloc[0]

t0_full = adc_df['t0'] 
t0_trimmed = adc_df_trimmed['t0'] 

fig = sp.make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=[
    "Piezo (raw data, full)", "Piezo (raw data, trimmed)", "Piezo (Filtered, trimmed)"
])

fig.add_trace(go.Scatter(
    x=t0_full, y=adc_df['adc0'], mode='lines', name='Piezo (raw data, full)'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=t0_trimmed, y=adc_df_trimmed['adc0'], mode='lines', name='Piezo (raw data, trimmed)'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=t0_trimmed, y=adc_df_trimmed['adc0_filtered'], mode='lines', name='Piezo (Filtered, trimmed)'
), row=3, col=1)

fig.update_layout(
    height=900,
    xaxis_title="Time (s)",
    yaxis_title="ADC0 Value",
    showlegend=False
)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.show()